# Tackling Overfitting With Dropout

Our baseline network overfit heavily - reaching 100% training accuracy but only around 92% on validation data. Here we add Dropout to the baseline, keeping everything else the same, so we can measure its effect on its own.

In [ ]:
# reload src/config.py automatically whenever it changes
%load_ext autoreload
%autoreload 2

# import project settings
import sys
sys.path.append("..")
from src import config

# import required packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from src.plots import plot_training_history
from pathlib import Path
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
# make results repeatable
tf.keras.utils.set_random_seed(config.RANDOM_SEED)

## Data Pipeline

Load images from the training, validation & test folders.

In [ ]:
# training data
training_set = tf.keras.utils.image_dataset_from_directory(
    config.TRAINING_DIR,
    image_size = (config.IMG_HEIGHT,config.IMG_WIDTH),
    batch_size = config.BATCH_SIZE,
    label_mode = 'categorical',
    shuffle = True,
    seed = config.RANDOM_SEED
)

# validation data
validation_set = tf.keras.utils.image_dataset_from_directory(
    config.VALIDATION_DIR,
    image_size = (config.IMG_HEIGHT,config.IMG_WIDTH),
    batch_size = config.BATCH_SIZE,
    label_mode = 'categorical',
    shuffle = False
    
)

# test data
test_set = tf.keras.utils.image_dataset_from_directory(
    config.TEST_DIR,
    image_size = (config.IMG_HEIGHT, config.IMG_WIDTH),
    label_mode = 'categorical',
    batch_size = config.BATCH_SIZE,
    shuffle = False
)

# class names
class_names = training_set.class_names
print(class_names)

## Network Architecture

The same architecture as the baseline, with one addition: a Dropout layer after the Dense layer, which randomly switches off 50% of its neurons during training.

In [ ]:
# network architecture (Functional API) - baseline + Dropout

# input: image of 80 x 60 pixels, 3 color channels
inputs = tf.keras.Input(shape=(config.IMG_HEIGHT,config.IMG_WIDTH, config.NUM_CHANNELS))

# rescale pixel values from 0-255 to 0-1
x = tf.keras.layers.Rescaling(1./255) (inputs)

# convolutional block 1
x = tf.keras.layers.Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu')(x)
x = tf.keras.layers.MaxPooling2D()(x)

# convolutional block 2
x = tf.keras.layers.Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu')(x)
x = tf.keras.layers.MaxPooling2D()(x)

# flatten the feature maps into one long list of numbers
x = tf.keras.layers.Flatten()(x)

# dense layer
x = tf.keras.layers.Dense(32, activation='relu')(x)

# dropout - randomly switch off 50% of the dense layer's neurons during training
x = tf.keras.layers.Dropout(0.5)(x)

# output layer
outputs = tf.keras.layers.Dense(config.NUM_CLASSES, activation='softmax')(x)

# connect layers
model = tf.keras.Model(inputs=inputs, outputs = outputs, name='dropout_cnn')

# view network architecture
model.summary()


In [ ]:
# compile the model
model.compile(loss = 'categorical_crossentropy',
             optimizer = 'adam',
             metrics = ['accuracy'])


## Training The Network

Train for 50 epochs, saving the best model (highest validation accuracy) to a new file so the baseline model is kept for comparison.

In [ ]:
# save the best model
model_filename = config.MODELS_DIR/"fashion_cnn_v02_dropout.keras"

save_best_model = tf.keras.callbacks.ModelCheckpoint(
    filepath=model_filename,
    monitor= 'val_accuracy',
    mode= 'max',
    save_best_only= True,
    verbose=0
)

# train the network
history = model.fit(
    training_set,
    validation_data=validation_set,
    epochs=config.NUM_EPOCHS,
    callbacks= [save_best_model],
    shuffle= False
)

## Analysis Of Training Results

Compare training & validation performance epoch by epoch - has the gap seen in the baseline been reduced?

In [ ]:
# plot training results
plot_training_history(history,
                      title="Dropout Network",
                      save_path=config.FIGURES_DIR / "dropout-accuracy-plot.png")

## Performance On The Test Set

Measure how the best saved model performs on images it has never seen, overall and for each class.

In [ ]:
# load the best model saved during training (highest validation accuracy)
best_model = tf.keras.models.load_model(model_filename)

# evaluate on the test set
test_loss, test_accuracy = best_model.evaluate(test_set, verbose=0)

print(f"Test accuracy: {test_accuracy:.2%}")
print(f"Test loss: {test_loss:.4f}")

In [ ]:
# predicted probabilities for every test image, one column per class
probabilities_df = pd.DataFrame(best_model.predict(test_set, verbose=0), columns=class_names)

# true labels for every test image (same order, because test_set has shuffle=False)
actual_df = pd.DataFrame(np.concatenate([labels for _, labels in test_set]), columns=class_names)

# create dataframe to analyse
predictions_df = pd.DataFrame({"actual_label": actual_df.idxmax(axis=1),
                               "predicted_label": probabilities_df.idxmax(axis=1),
                               "predicted_probability": probabilities_df.max(axis=1),
                               "filename": [Path(path).name for path in test_set.file_paths]})

predictions_df["correct"] = np.where(predictions_df["actual_label"] == predictions_df["predicted_label"], 1, 0)


print(predictions_df["correct"].mean())

In [ ]:


# confusion matrix heatmap - rows = actual class, columns = predicted class
fig, ax = plt.subplots(figsize=(8, 7))

ConfusionMatrixDisplay.from_predictions(
    predictions_df["actual_label"],
    predictions_df["predicted_label"],
    labels=config.CLASSES,       # look-alike pairs next to each other
    cmap="Blues",
    colorbar=False,
    ax=ax
)

ax.set_title("Dropout Network - Test Set Confusion Matrix")
ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "dropout-confusion-matrix.png", dpi=150)
plt.show()

# accuracy for each class
class_accuracy = predictions_df.groupby("actual_label")["correct"].mean().sort_values()
print(class_accuracy.round(2))

## Error Analysis

Look at the network's mistakes and how confident it was - and test whether that confidence could be used to decide which products to auto-tag.

In [ ]:
# the model's mistakes, most confident first
mistakes = predictions_df.loc[predictions_df["correct"] == 0].sort_values("predicted_probability", ascending=False)
print(f"Number of mistakes: {len(mistakes)}")

# typical confidence when right (1) vs wrong (0)
print(predictions_df.groupby("correct")["predicted_probability"].median())

mistakes.head(10)

In [ ]:
# what if we only auto-tag products the model is at least 90% confident?
threshold = 0.90

confident = predictions_df.loc[predictions_df["predicted_probability"] >= threshold]
not_confident = predictions_df.loc[predictions_df["predicted_probability"] < threshold]

print(f"Auto-tagged: {len(confident)} of {len(predictions_df)} products ({len(confident) / len(predictions_df):.0%})")
print(f"Accuracy on auto-tagged products: {confident['correct'].mean():.1%}")
print(f"Sent to a person to check: {len(not_confident)} products")

## Summary

**Change from Baseline**

* Added a Dropout layer (rate 0.5) after the Dense layer - everything else unchanged
* Parameters unchanged at 317,574 (Dropout has nothing to learn)

**Training**

* Best validation accuracy: **92.4%** at epoch 38
* Overfitting largely removed - training & validation accuracy finish together at around 91% (baseline: 100% vs 92%)
* Validation loss stays around 0.3 (baseline: rose to 0.59). Mild overfitting appears from around epoch 30

**Test Set Performance**

| | Baseline | Dropout |
|---|---|---|
| Test accuracy | 92.3% | **91.5%** |
| Test loss | 0.58 | **0.38** |

| Class | Baseline | Dropout |
|---|---|---|
| Handbags | 99% | 99% |
| Watches | 98% | 97% |
| Shirts | 95% | 95% |
| Tshirts | 94% | 94% |
| Casual Shoes | 89% | 85% |
| Sports Shoes | 79% | 79% |

* Accuracy is effectively the same as the baseline - 549 vs 554 correct, a difference of just 5 images out of 600
* Test loss fell by around a third, so the network's confidence is more realistic
* Sports Shoes remain the weakest class, still most often confused with Casual Shoes
* Several of the same images were misclassified by both networks (e.g. 22543.jpg, 4133.jpg, 19595.jpg), supporting the label noise finding from the baseline

**Error Analysis**

* Median confidence is 99.99% on correct predictions vs 84% on mistakes - so confidence partly separates right from wrong
* However, the most confident mistakes are still close to 100%, so some overconfidence remains

**Business Application**

* Auto-tagging only products with at least 90% confidence covers 88% of products at 96.0% accuracy, with the remaining 12% (69 products) sent for manual review
* In practice, this threshold would be chosen using the validation set

**Next:** `04_image_augmentation.ipynb` - add Image Augmentation to the training pipeline